# Heat 1D · DeepXDE v1.12.2 demo (Kaggle)

Reproducción remota del demo oficial con cinco seeds y referencia analítica. Requiere Kaggle GPU T4×2. DeepXDE 1.12.2 se instala solo si falta el pin exacto.


## Protocolo

- Heat 1D en [0,1] × [0,1]: u_t = 0.1 u_xx, Dirichlet homogénea y u(x,0)=sin(πx).
- Red [2,20,20,20,1], tanh, Glorot normal; 2540 puntos interiores, 80 frontera, 160 iniciales, muestreo Hammersley.
- Adam lr=1e-3, 20 000 iteraciones, seguido de L-BFGS-B.
- Seeds [901,902,903,904,905]; MAE, RMSE, L2 relativo y residual medio absoluto en 512×256 y 256×201.
- Referencia exacta: exp(-0.1 π² t) sin(πx).

Este brazo conserva la receta DeepXDE 1.12.2 (red, puntos y optimizador), pero fija α=0.1 para igualar el contrato Heat de AF_PIKANs. Reporta también la grilla de resolución del demo (256×201); el demo oficial usa α=0.4, por lo que las métricas de ambos coeficientes no se mezclan.


In [ ]:

import csv,gc,importlib.metadata as md,json,os,platform,random,signal,subprocess,sys,time,traceback
from datetime import datetime,timezone
from pathlib import Path
os.environ["DDE_BACKEND"]="pytorch"
os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG",":4096:8")
OUT=Path("/kaggle/working/heat_deepxde_reference")
JP,CP=OUT/"run_summary.json",OUT/"seed_metrics.csv"
VERSION="1.12.2"; SEEDS=[901,902,903,904,905]; A=.1; HARD=1000; WATCH=950
MAIN=(512,256); DEMO=(256,201); ADAM=20000
FIELDS=["seed","status","phase","elapsed_s","adam_s","lbfgsb_s","eval_s",
 "adam_steps","lbfgsb_steps","main_rel_l2","main_mae","main_rmse","main_abs_residual",
 "demo_rel_l2","demo_mae","demo_rmse","demo_abs_residual","error"]
class WatchdogExpired(Exception): pass
def alarm(_s,_f): raise WatchdogExpired("notebook watchdog 950 s")
def utc(): return datetime.now(timezone.utc).isoformat()
def put_json(path,obj):
 tmp=path.with_suffix(path.suffix+".tmp")
 with tmp.open("w",encoding="utf-8") as f:
  json.dump(obj,f,ensure_ascii=False,indent=2);f.write("\\n");f.flush();os.fsync(f.fileno())
 os.replace(tmp,path)
def put_csv(path,rows):
 tmp=path.with_suffix(path.suffix+".tmp")
 with tmp.open("w",encoding="utf-8",newline="") as f:
  w=csv.DictWriter(f,fieldnames=FIELDS);w.writeheader()
  for r in rows:w.writerow({k:r.get(k,"") for k in FIELDS})
  f.flush();os.fsync(f.fileno())
 os.replace(tmp,path)
t0=time.monotonic();deadline=t0+WATCH;started=utc()
OUT.mkdir(parents=True,exist_ok=True);rows=[]
active={"row":None,"seed_t":None,"phase_t":None,"phase":None}
state={"schema_version":1,"benchmark":"heat-1d-deepxde-v1.12.2-alpha-0.1","status":"starting",
 "stage":"startup","stop_reason":None,"started_at_utc":started,"ended_at_utc":None,
 "elapsed_seconds":0.,"configuration":{"deepxde":"1.12.2","backend":"pytorch","alpha":A,
 "domain":[0,1],"time":[0,1],"initial":"sin(pi*x)","boundary":"homogeneous Dirichlet",
 "seeds":SEEDS,"architecture":[2,20,20,20,1],"activation":"tanh","initializer":"Glorot normal",
 "num_domain":2540,"num_boundary":80,"num_initial":160,"sampling":"Hammersley",
 "adam_lr":1e-3,"adam_iterations":ADAM,"lbfgs":"L-BFGS-B","main_grid_space_time":MAIN,
 "official_demo_grid_space_time":DEMO,"official_demo_alpha":.4,
 "demo_note":"Exact DeepXDE v1.12.2 demo physics and method at alpha=0.4."},
 "budget":{"hard_limit_seconds":HARD,"watchdog_seconds":WATCH,"cli_timeout_seconds":HARD,
 "timer_starts_before_install_and_training":True},
 "provenance":{"python":platform.python_version(),"platform":platform.platform(),
 "deepxde":None,"torch":None,"installation":None,"gpu_names":[],"gpu_count":0,
 "two_t4s_validated":False,"model_device":"cuda:0",
 "second_t4":"unused by each single-device DeepXDE/PyTorch run","reference_protocol":"official DeepXDE 1.12.2 Heat demo","local_alpha_0_1_pilot":"separate arm; not included or directly compared"},
 "resource_estimate":{},"runs":[]}
def save(stage=None,status=None,reason=None):
 if stage is not None:state["stage"]=stage
 if status is not None:state["status"]=status
 if reason is not None:state["stop_reason"]=str(reason)
 state["elapsed_seconds"]=round(time.monotonic()-t0,3)
 state["runs"]=[dict(r) for r in rows]
 if state["provenance"]["gpu_count"]:
  e=state["elapsed_seconds"];n=state["provenance"]["gpu_count"]
  state["resource_estimate"]={"gpu_wall_hours":round(e/3600,6),
   "two_device_hours_if_both_counted":round(e*n/3600,6),
   "note":"Elapsed estimate; Kaggle quota accounting is external."}
 put_json(JP,state);put_csv(CP,rows)
def phase(seed,name):
 active.update(phase=name,phase_t=time.monotonic())
 active["row"].update(status="running",phase=name);save(f"seed_{seed}_{name}","running")
def end_phase(field):
 active["row"][field]=round(time.monotonic()-active["phase_t"],3)
 active["phase_t"]=None;save()
def exact(x,t):
 import numpy as np
 return np.exp(-A*np.pi**2*t)*np.sin(np.pi*x)
def pde(x,y):
 return dde.grad.jacobian(y,x,i=0,j=1)-A*dde.grad.hessian(y,x,i=0,j=0)
def metrics(model,nx,nt,np,chunk=2048):
 xs=np.linspace(0,1,nx,dtype=np.float32);ts=np.linspace(0,1,nt,dtype=np.float32)
 tt,xx=np.meshgrid(ts,xs,indexing="ij")
 X=np.column_stack((xx.ravel(),tt.ravel())).astype(np.float32)
 y=exact(X[:,0:1],X[:,1:2]);pred=[];res=[]
 for i in range(0,len(X),chunk):
  b=X[i:i+chunk];pred.append(np.asarray(model.predict(b)).ravel())
  res.append(np.abs(np.asarray(model.predict(b,operator=pde)).ravel()))
 err=np.concatenate(pred)-y.ravel();res=np.concatenate(res)
 return {"rel_l2":float(np.linalg.norm(err)/np.linalg.norm(y)),
  "mae":float(np.mean(np.abs(err))),"rmse":float(np.sqrt(np.mean(err**2))),
  "abs_residual":float(np.mean(res))}
def install():
 try:v=md.version("deepxde")
 except md.PackageNotFoundError:v=None
 if v=="1.12.2":state["provenance"]["installation"]="preinstalled_exact_pin";return
 state["provenance"]["installation"]="conditional_install_started";save("installing_deepxde","running")
 try:subprocess.run([sys.executable,"-m","pip","install","--quiet","--disable-pip-version-check",
  "deepxde==1.12.2"],check=True,timeout=600)
 except subprocess.TimeoutExpired as e:raise WatchdogExpired("conditional install timed out") from e
 state["provenance"]["installation"]="installed_exact_pin"
def seed_all(seed,np,torch):
 dde.config.set_random_seed(seed);random.seed(seed);np.random.seed(seed)
 torch.manual_seed(seed);torch.cuda.manual_seed_all(seed)
 torch.backends.cudnn.deterministic=True;torch.backends.cudnn.benchmark=False
 torch.use_deterministic_algorithms(True,warn_only=True)
signal.signal(signal.SIGALRM,alarm)
signal.setitimer(signal.ITIMER_REAL,max(.001,deadline-time.monotonic()))
try:
 save("startup","running")
 if not Path("/kaggle/working").is_dir():raise RuntimeError("Kaggle remote runtime required")
 setup_t=time.monotonic();install()
 import numpy as np
 import torch
 import deepxde as dde
 if dde.__version__!=VERSION:raise RuntimeError(f"Expected DeepXDE {VERSION}, got {dde.__version__}")
 if dde.backend.backend_name!="pytorch":raise RuntimeError("DeepXDE backend must be pytorch")
 if not torch.cuda.is_available():raise RuntimeError("Kaggle T4x2 required; CUDA unavailable")
 names=[torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())]
 if len(names)!=2 or not all("T4" in n.upper() for n in names):
  raise RuntimeError(f"Kaggle T4x2 required; visible GPUs: {names}")
 torch.cuda.set_device(0);dde.config.set_default_float("float32")
 state["provenance"].update(deepxde=dde.__version__,torch=torch.__version__,
  gpu_names=names,gpu_count=len(names),two_t4s_validated=True)
 setup_s=round(time.monotonic()-setup_t,3);save("gpu_validated","running")
 print(json.dumps({"deepxde":dde.__version__,"torch":torch.__version__,"gpus":names}))
 for seed in SEEDS:
  if time.monotonic()>=deadline:raise WatchdogExpired("no budget for next seed")
  r={k:"" for k in FIELDS};r.update(seed=seed,status="running",phase="build",adam_steps=ADAM)
  rows.append(r);active.update(row=r,seed_t=time.monotonic(),phase="build",phase_t=None)
  seed_all(seed,np,torch);save(f"seed_{seed}_build","running")
  geom=dde.geometry.Interval(0,1);td=dde.geometry.TimeDomain(0,1)
  gt=dde.geometry.GeometryXTime(geom,td)
  bc=dde.icbc.DirichletBC(gt,lambda x:np.zeros((len(x),1),dtype=np.float32),
   lambda x,on_boundary:on_boundary)
  ic=dde.icbc.IC(gt,lambda x:np.sin(np.pi*x[:,0:1]),lambda x,on_initial:on_initial)
  data=dde.data.TimePDE(gt,pde,[bc,ic],num_domain=2540,num_boundary=80,
   num_initial=160,train_distribution="Hammersley",num_test=2540)
  net=dde.nn.FNN([2,20,20,20,1],"tanh","Glorot normal");model=dde.Model(data,net)
  phase(seed,"adam");model.compile("adam",lr=1e-3)
  ah,ts=model.train(iterations=ADAM,display_every=1000);end_phase("adam_s")
  phase(seed,"lbfgsb");model.compile("L-BFGS-B")
  lh,ts=model.train();end_phase("lbfgsb_s")
  r["lbfgsb_steps"]=max(0,int(getattr(ts,"step",ADAM))-ADAM)
  phase(seed,"evaluation");m=metrics(model,*MAIN,np);d=metrics(model,*DEMO,np);end_phase("eval_s")
  for prefix,result in (("main",m),("demo",d)):
   r[prefix+"_rel_l2"]=result["rel_l2"];r[prefix+"_mae"]=result["mae"]
   r[prefix+"_rmse"]=result["rmse"];r[prefix+"_abs_residual"]=result["abs_residual"]
  r.update(status="complete",phase="complete",elapsed_s=round(time.monotonic()-active["seed_t"],3))
  save(f"seed_{seed}_complete","running")
  print(json.dumps({"seed":seed,"main_512x256":m,"demo_256x201":d}))
  del model,net,data,ts,ah,lh;gc.collect();torch.cuda.empty_cache()
 state["ended_at_utc"]=utc();save("complete","complete")
 print(json.dumps({"status":state["status"],"elapsed_seconds":state["elapsed_seconds"],
  "json":str(JP),"csv":str(CP)}))
except WatchdogExpired as e:
 if active["row"] is not None:
  r=active["row"];field={"adam":"adam_s","lbfgsb":"lbfgsb_s","evaluation":"eval_s"}.get(active["phase"])
  if field and active["phase_t"] is not None:
   r[field]=round(float(r.get(field) or 0)+time.monotonic()-active["phase_t"],3)
  r.update(status="truncated",phase=active["phase"],error=str(e),
   elapsed_s=round(time.monotonic()-active["seed_t"],3))
 state["ended_at_utc"]=utc();save("truncated","truncated",str(e))
 print(f"TRUNCATED: {e}; partial state: {JP} and {CP}")
except Exception as e:
 if active["row"] is not None:
  active["row"].update(status="failed",phase=active["phase"],error=f"{type(e).__name__}: {e}")
 state["ended_at_utc"]=utc();state["error"]={"type":type(e).__name__,"message":str(e),
  "traceback":traceback.format_exc()[-5000:]}
 save("failed","failed",f"{type(e).__name__}: {e}");raise
finally:
 signal.setitimer(signal.ITIMER_REAL,0)


## Presupuesto, hardware y ejecución

El watchdog corta a 950 s desde el inicio de la primera celda, incluidos instalación condicional, imports, entrenamiento, evaluación y escritura. Techo CLI: 1000 s. Deja status truncated, seed/fase, costo parcial y provenance en /kaggle/working/heat_deepxde_reference/run_summary.json, más CSV parcial seed_metrics.csv.

La metadata solicita NvidiaTeslaT4 (Kaggle T4×2) y valida dos T4 visibles. DeepXDE/PyTorch corre cada modelo en cuda:0; la segunda T4 queda sin uso por corrida. El JSON estima GPU-horas de pared y dispositivo-horas si se cuentan ambas tarjetas.

Para reservar al menos 12 h semanales, usar el techo conservador de 4 h de dispositivo por corrida de hasta 2 h: ejecutar solo con saldo previo de al menos 16 h. Este bundle no consulta ni consume cuota.

Reemplaza el username en kernel-metadata.json y ejecuta:

    kaggle kernels push -p var/agent_runs/baseline-competitive-2026-09-27/followup-20260929/heat/deepxde-paired-901-905 --accelerator NvidiaTeslaT4 --timeout 1000

Sin datasets ni kernels fuente. Internet se usa solo si hay que descargar el pin DeepXDE. Referencias: [demo oficial](https://deepxde.readthedocs.io/en/v1.12.2/demos/pinn_forward/heat.html), [metadata Kaggle](https://github.com/Kaggle/kaggle-cli/blob/main/docs/kernels_metadata.md), [CLI](https://github.com/Kaggle/kaggle-cli/blob/main/docs/kernels.md).
